# QLoRA clean pilot: 2k/5k train on `instruction_temporal`

Goal: test QLoRA/rec-tuning on the cleaned instruction dataset after an overfit sanity check.

Key settings:
- use prepared `instruction_temporal/train.jsonl`, `val.jsonl`, and `test.jsonl` files;
- do not rebuild prompts from `item_text_retrieval`;
- compute loss only on the `Yes/No` answer;
- check truncation behavior;
- save adapter, predictions, and metrics;
- start from pilot-size training instead of the full 80k split.


## Input files and run settings

Expected `instruction_temporal` files:

```text
train.jsonl
val.jsonl
test.jsonl
```

Initial pilot mode:

```text
TRAIN_PER_CLASS = 1000  # 2000 train examples total
EVAL_PER_CLASS = 500    # 1000 validation and 1000 test examples
NUM_EPOCHS = 3
```

Extended pilot mode:

```text
TRAIN_PER_CLASS = 2500  # 5000 train examples total
```


In [ ]:
!pip install -q -U "transformers>=4.45.0" accelerate peft bitsandbytes datasets scikit-learn pandas pyarrow


In [ ]:
import os, json, random, math, shutil, glob, gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("device count:", torch.cuda.device_count())


## 1. Конфиг

Сначала не гоним full train. Нам нужен нормальный pilot, который покажет, обобщается ли QLoRA после overfit200.


In [ ]:
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
# Если 4B не влезает, можно заменить:
# MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

DATA_ROOT = None  # None = искать train/val/test.jsonl внутри /kaggle/input

MAX_SEQ_LENGTH = 1024
TRAIN_PER_CLASS = 2500   # 1000 -> 2k train; 2500 -> 5k train
EVAL_PER_CLASS = 1000     # 500 -> 1k val/test
TRAIN_EVAL_PER_CLASS = 500

NUM_EPOCHS = 2
LR = 2e-4
GRAD_ACCUM = 8
BATCH_SIZE = 1
SAVE_STEPS = 250

RUN_NAME = f"qwen_clean_pilot_train{TRAIN_PER_CLASS*2}_eval{EVAL_PER_CLASS*2}"
OUT_DIR = Path("/kaggle/working") / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("run:", RUN_NAME)
print("out:", OUT_DIR)


In [ ]:
def find_file(filename):
    hits = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"Не найден {filename} в /kaggle/input. Проверь, что Kaggle Dataset подключён.")
    # Предпочитаем instruction_temporal, если есть несколько совпадений
    hits = sorted(hits, key=lambda p: (("instruction_temporal" not in p.lower()), len(p)))
    return Path(hits[0])

if DATA_ROOT is None:
    train_path = find_file("train.jsonl")
    val_path = find_file("val.jsonl")
    test_path = find_file("test.jsonl")
else:
    root = Path(DATA_ROOT)
    train_path = root / "train.jsonl"
    val_path = root / "val.jsonl"
    test_path = root / "test.jsonl"

print("train:", train_path)
print("val:", val_path)
print("test:", test_path)


In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

train_df = load_jsonl(train_path)
val_df = load_jsonl(val_path)
test_df = load_jsonl(test_path)

print("shapes:", train_df.shape, val_df.shape, test_df.shape)
print("columns:", train_df.columns.tolist())
print("train output:", train_df["output"].value_counts(dropna=False).to_dict())
print("val output:", val_df["output"].value_counts(dropna=False).to_dict())
print("test output:", test_df["output"].value_counts(dropna=False).to_dict())


## 2. Balanced pilot samples

Важно: это pilot. Мы специально берём небольшую сбалансированную выборку, чтобы быстро понять, работает ли clean QLoRA.


In [ ]:
def make_balanced_sample(df, n_per_class, seed=42):
    out_col = df["output"].astype(str).str.strip().str.lower()
    yes = df[out_col == "yes"].sample(n=n_per_class, random_state=seed)
    no = df[out_col == "no"].sample(n=n_per_class, random_state=seed)
    return pd.concat([yes, no], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)

train_pilot_df = make_balanced_sample(train_df, TRAIN_PER_CLASS, SEED)
train_eval_df = make_balanced_sample(train_pilot_df, min(TRAIN_EVAL_PER_CLASS, TRAIN_PER_CLASS), SEED + 1)
val_pilot_df = make_balanced_sample(val_df, EVAL_PER_CLASS, SEED)
test_pilot_df = make_balanced_sample(test_df, EVAL_PER_CLASS, SEED)

for name, df in [
    ("train_pilot", train_pilot_df),
    ("train_eval", train_eval_df),
    ("val_pilot", val_pilot_df),
    ("test_pilot", test_pilot_df)
]:
    print(name, df.shape, df["output"].value_counts().to_dict())


## 3. Prompt builder

Используем готовый instruction dataset. Если есть `instruction` и `input`, склеиваем их. Если есть только `prompt_text`, используем его.


In [ ]:
def normalize_answer(x):
    x = str(x).strip()
    if x.lower().startswith("yes"):
        return "Yes"
    if x.lower().startswith("no"):
        return "No"
    raise ValueError(f"Bad output: {x}")

def build_user_text(row):
    instruction = str(row.get("instruction", "")).strip()
    inp = str(row.get("input", "")).strip()
    if instruction and inp:
        return instruction + "\n\n" + inp
    if "prompt_text" in row and pd.notna(row["prompt_text"]):
        return str(row["prompt_text"]).strip()
    raise ValueError("Нет instruction/input и нет prompt_text")

print(build_user_text(train_pilot_df.iloc[0])[:1800])
print("ANSWER:", normalize_answer(train_pilot_df.iloc[0]["output"]))


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

SYSTEM_PROMPT = "You are a recommendation model. Answer only Yes or No."

def make_prompt_text(row):
    user_text = build_user_text(row)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def make_answer_text(row):
    return normalize_answer(row["output"]) + tokenizer.eos_token

prompt_sample = make_prompt_text(train_pilot_df.iloc[0])
answer_sample = make_answer_text(train_pilot_df.iloc[0])
print(prompt_sample[-1500:])
print("answer:", repr(answer_sample))


## 4. Dataset с answer-only loss

Prompt tokens получают label `-100`, loss считается только на ответе `Yes/No`.


In [ ]:
class RecSFTDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_length=1024):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt_text = make_prompt_text(row)
        answer_text = make_answer_text(row)
        prompt_ids = self.tokenizer(prompt_text, add_special_tokens=False).input_ids
        answer_ids = self.tokenizer(answer_text, add_special_tokens=False).input_ids

        max_prompt_len = self.max_length - len(answer_ids)
        if max_prompt_len <= 0:
            raise ValueError("Answer longer than max_length")
        truncated = 0
        if len(prompt_ids) > max_prompt_len:
            truncated = len(prompt_ids) - max_prompt_len
            prompt_ids = prompt_ids[-max_prompt_len:]

        input_ids = prompt_ids + answer_ids
        labels = [-100] * len(prompt_ids) + answer_ids
        attention_mask = [1] * len(input_ids)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "truncated_tokens": truncated
        }

def collate_fn(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, attention_mask, labels = [], [], []
    for x in batch:
        pad_len = max_len - len(x["input_ids"])
        input_ids.append(torch.cat([x["input_ids"], torch.full((pad_len,), tokenizer.pad_token_id, dtype=torch.long)]))
        attention_mask.append(torch.cat([x["attention_mask"], torch.zeros(pad_len, dtype=torch.long)]))
        labels.append(torch.cat([x["labels"], torch.full((pad_len,), -100, dtype=torch.long)]))
    return {
        "input_ids": torch.stack(input_ids),
        "attention_mask": torch.stack(attention_mask),
        "labels": torch.stack(labels)
    }

train_ds = RecSFTDataset(train_pilot_df, tokenizer, MAX_SEQ_LENGTH)

lens, trunc = [], []
for i in range(min(len(train_ds), 1000)):
    item = train_ds[i]
    lens.append(len(item["input_ids"]))
    trunc.append(int(item["truncated_tokens"]))

print("length min/mean/max:", min(lens), float(np.mean(lens)), max(lens))
print("truncated examples in checked subset:", sum(t > 0 for t in trunc), "of", len(trunc))
print("max truncated tokens:", max(trunc))
print("non-ignored label tokens in sample:", int((train_ds[0]["labels"] != -100).sum()))


## 5. Загрузка модели и LoRA


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 6. Evaluation через logprob `Yes` vs `No`

Считаем вероятность двух ответов:

```text
score = P(Yes) / (P(Yes) + P(No))
```

Используем logprob всей последовательности ответа, а не один первый токен.


In [ ]:
@torch.no_grad()
def answer_logprob(model, prompt_text, answer_text):
    model.eval()
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False).input_ids
    answer_ids = tokenizer(answer_text, add_special_tokens=False).input_ids
    max_prompt_len = MAX_SEQ_LENGTH - len(answer_ids)
    if len(prompt_ids) > max_prompt_len:
        prompt_ids = prompt_ids[-max_prompt_len:]
    input_ids = torch.tensor([prompt_ids + answer_ids], dtype=torch.long, device=model.device)
    attention_mask = torch.ones_like(input_ids)
    out = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = out.logits[0]
    start = len(prompt_ids)
    total = 0.0
    for j, tok_id in enumerate(answer_ids):
        pos = start + j - 1
        log_probs = torch.log_softmax(logits[pos], dim=-1)
        total += float(log_probs[tok_id].detach().cpu())
    return total

@torch.no_grad()
def predict_scores(model, df):
    scores, y_true, sample_ids = [], [], []
    yes_answer = "Yes" + tokenizer.eos_token
    no_answer = "No" + tokenizer.eos_token
    for _, row in df.iterrows():
        prompt_text = make_prompt_text(row)
        lp_yes = answer_logprob(model, prompt_text, yes_answer)
        lp_no = answer_logprob(model, prompt_text, no_answer)
        m = max(lp_yes, lp_no)
        p_yes = math.exp(lp_yes - m) / (math.exp(lp_yes - m) + math.exp(lp_no - m))
        scores.append(p_yes)
        y_true.append(1 if normalize_answer(row["output"]) == "Yes" else 0)
        sample_ids.append(row.get("sample_id", None))
    return np.array(y_true), np.array(scores), sample_ids

def best_threshold_by_f1(y_true, scores):
    best = {"threshold": 0.5, "f1": -1}
    for thr in np.linspace(0.01, 0.99, 99):
        pred = (scores >= thr).astype(int)
        p, r, f1, _ = precision_recall_fscore_support(y_true, pred, average="binary", zero_division=0)
        if f1 > best["f1"]:
            best = {"threshold": float(thr), "precision": float(p), "recall": float(r), "f1": float(f1)}
    return best

def calc_metrics(y_true, scores, threshold):
    pred = (scores >= threshold).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_true, pred, average="binary", zero_division=0)
    return {
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "pr_auc": float(average_precision_score(y_true, scores)),
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(p),
        "recall": float(r),
        "f1": float(f1),
        "threshold": float(threshold)
    }

def evaluate_split(model, df, name, threshold=None):
    y_true, scores, sample_ids = predict_scores(model, df)
    if threshold is None:
        threshold = best_threshold_by_f1(y_true, scores)["threshold"]
    metrics = calc_metrics(y_true, scores, threshold)
    print(name, metrics)
    pred_df = pd.DataFrame({
        "sample_id": sample_ids,
        "y_true": y_true,
        "score_yes": scores,
        "pred": (scores >= threshold).astype(int)
    })
    return metrics, pred_df


## 7. Base model metrics до обучения

Это не обязательно, но полезно: покажет стартовую точку на тех же pilot split.


In [ ]:
base_val_metrics, base_val_pred = evaluate_split(model, val_pilot_df, "base_val_pilot")
base_thr = base_val_metrics["threshold"]
base_train_metrics, base_train_pred = evaluate_split(model, train_eval_df, "base_train_eval", threshold=base_thr)
base_test_metrics, base_test_pred = evaluate_split(model, test_pilot_df, "base_test_pilot", threshold=base_thr)


## 8. Обучение QLoRA pilot


In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUT_DIR / "trainer"),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    fp16=True,
    logging_steps=20,
    save_steps=SAVE_STEPS,
    save_strategy="steps",
    save_total_limit=3,
    report_to="none",
    optim="paged_adamw_8bit",
    seed=SEED,
    remove_unused_columns=False,
    gradient_checkpointing=True,
    warmup_ratio=0.03
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=collate_fn
)

print("train examples:", len(train_ds))
print("approx optimization steps:", math.ceil(len(train_ds) / (BATCH_SIZE * GRAD_ACCUM)) * NUM_EPOCHS)
trainer.train()


## 9. Метрики после обучения

Threshold подбирается только на val, затем фиксируется для train_eval и test.


In [ ]:
qlora_val_metrics, qlora_val_pred = evaluate_split(model, val_pilot_df, "qlora_val_pilot")
thr = qlora_val_metrics["threshold"]
qlora_train_metrics, qlora_train_pred = evaluate_split(model, train_eval_df, "qlora_train_eval", threshold=thr)
qlora_test_metrics, qlora_test_pred = evaluate_split(model, test_pilot_df, "qlora_test_pilot", threshold=thr)

summary = pd.DataFrame([
    {"stage": "base", "split": "train_eval", **base_train_metrics},
    {"stage": "base", "split": "val_pilot", **base_val_metrics},
    {"stage": "base", "split": "test_pilot", **base_test_metrics},
    {"stage": "qlora", "split": "train_eval", **qlora_train_metrics},
    {"stage": "qlora", "split": "val_pilot", **qlora_val_metrics},
    {"stage": "qlora", "split": "test_pilot", **qlora_test_metrics},
])
summary


In [ ]:
summary.to_csv(OUT_DIR / "qwen_clean_pilot_metrics.csv", index=False)
base_val_pred.to_csv(OUT_DIR / "base_val_pilot_predictions.csv", index=False)
base_test_pred.to_csv(OUT_DIR / "base_test_pilot_predictions.csv", index=False)
qlora_train_pred.to_csv(OUT_DIR / "qlora_train_eval_predictions.csv", index=False)
qlora_val_pred.to_csv(OUT_DIR / "qlora_val_pilot_predictions.csv", index=False)
qlora_test_pred.to_csv(OUT_DIR / "qlora_test_pilot_predictions.csv", index=False)

run_summary = {
    "model_name": MODEL_NAME,
    "run_name": RUN_NAME,
    "train_examples": int(len(train_pilot_df)),
    "val_examples": int(len(val_pilot_df)),
    "test_examples": int(len(test_pilot_df)),
    "train_per_class": int(TRAIN_PER_CLASS),
    "eval_per_class": int(EVAL_PER_CLASS),
    "num_epochs": int(NUM_EPOCHS),
    "lr": float(LR),
    "max_seq_length": int(MAX_SEQ_LENGTH),
    "metrics": summary.to_dict(orient="records")
}
with open(OUT_DIR / "qwen_clean_pilot_summary.json", "w", encoding="utf-8") as f:
    json.dump(run_summary, f, ensure_ascii=False, indent=2)

print("saved to", OUT_DIR)


## 10. Generation format check

This section checks that the model generates only `Yes` or `No`.


In [ ]:
@torch.no_grad()
def generate_answer(model, row, max_new_tokens=4):
    model.eval()
    prompt_text = make_prompt_text(row)
    inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return gen.strip()

examples = []
for i in range(min(20, len(test_pilot_df))):
    row = test_pilot_df.iloc[i]
    examples.append({
        "i": i,
        "true": normalize_answer(row["output"]),
        "generated": generate_answer(model, row)
    })
gen_df = pd.DataFrame(examples)
gen_df.to_csv(OUT_DIR / "generated_test_examples.csv", index=False)
gen_df


## 11. Сохранение adapter и архивов


In [ ]:
adapter_dir = OUT_DIR / "adapter_qwen_clean_pilot"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

shutil.make_archive(str(OUT_DIR / "adapter_qwen_clean_pilot"), "zip", adapter_dir)
shutil.make_archive(str(OUT_DIR / "qwen_clean_pilot_full_outputs"), "zip", OUT_DIR)

print("adapter zip:", OUT_DIR / "adapter_qwen_clean_pilot.zip")
print("full outputs zip:", OUT_DIR / "qwen_clean_pilot_full_outputs.zip")


# Как интерпретировать

## Хорошо

Если QLoRA на `test_pilot` даёт ROC-AUC хотя бы около `0.65-0.75`, значит clean instruction pipeline даёт существенный прирост над старым плохим QLoRA запуском.

## Средне

Если ROC-AUC около `0.57-0.63`, pipeline работает, но данных/эпох/модели мало. Можно попробовать 5k train или лучше модель.

## Плохо

Если ROC-AUC снова около `0.5`, несмотря на хороший overfit200, значит модель хорошо запоминает маленькую выборку, но плохо обобщается в этой постановке. Тогда mainline лучше держать на semantic features + CatBoost, а QLoRA оставить как отрицательный/ограниченный результат.
